In [1]:
# This notebook sets up and trains Deepdisc (Merz et al. 2023) on the Cen V854 plates, as a test run.

# [Setup]: 

# Ran the following in Anaconda Prompt:
# git clone https://github.com/burke86/deepdisc.git
# cd deepdisc
# conda create -n deepdisc python=3.10 -y
# conda activate deepdisc
# pip install setuptools==67.8.0
# pip install pybind11
# NOTE: scarlet skipped, does not compile on Windows (optional dependency, not needed for detection)

# Ran the following in Anaconda Prompt:
# nvidia-smi

# If Version CUDA is not 12.1, install older versions to be compatible with torch and relevant packages:
# Ran the following in Anaconda Prompt:
# conda activate deepdisc
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Ran the following in Anaconda Prompt:
# pip install --no-build-isolation git+https://github.com/facebookresearch/detectron2.git
# mkdir taufit
# echo __version__ = "0.1.0" > taufit\version.py
# echo. > requirements.txt
# pip install --no-build-isolation -e .
# pip install --no-deps --no-build-isolation .
# pip install opencv-python
# pip install astropy photutils matplotlib pandas ipykernel
# python -m ipykernel install --user --name deepdisc --display-name "Python (deepdisc)"

# Now, in VSCodium, click top right for environment -> "Choose another Kernel" -> "deepdisc"

In [ ]:
# Pytorch trial run Cell 1

# NOTE: Running this will take a while as COCO generates an image for each .fits file.

from pathlib import Path
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from photutils.detection import find_peaks
import numpy as np
import cv2
import json
import random

# Project paths

BASE_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN")

cutout_dir = BASE_DIR / "data" / "raw" / "v854_cen_dasch" / "cutouts"

output_dir = BASE_DIR / "data" / "PyTorch"
img_dir   = output_dir / "images"
train_dir = output_dir / "train"
val_dir   = output_dir / "val"

img_dir.mkdir(parents=True, exist_ok=True)
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

# Checks input data
print("Looking in:", cutout_dir)
print("Exists:", cutout_dir.exists())

cutouts = sorted(list(cutout_dir.glob("*.fits")) + list(cutout_dir.glob("*.fit")))

print("Num FITS files:", len(cutouts))

if len(cutouts) == 0:
    print("\nNo FITS files found. Debug listing:")
    if cutout_dir.exists():
        for p in cutout_dir.rglob("*"):
            print(p)
    raise RuntimeError("No FITS files found in cutout directory")

# Parameters
THRESHOLD_SIGMA = 2.5
BOX_SIZE = 11
BOX_HALF = 8
VAL_SPLIT = 0.15

# Coco structure
coco_all = {
    "images": [],
    "annotations": [],
    "categories": [{"id": 1, "name": "star"}]
}

ann_id = 0
coco_id = 0
valid_ids = []

# Main loop
for fits_path in cutouts:

    try:
        # Load FITS
        data = fits.getdata(fits_path).astype(float)
        data = np.nan_to_num(data, nan=np.nanmedian(data))

        mean, median, std = sigma_clipped_stats(data, sigma=3.0)
        data_sub = data - median

        # Detect sources
        try:
            sources = find_peaks(
                data_sub,
                threshold=THRESHOLD_SIGMA * std,
                box_size=BOX_SIZE
            )
        except Exception:
            sources = None

        # Normalize image
        vmin, vmax = np.percentile(data, [1, 99])
        img_norm = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
        img_8bit = (img_norm * 255).astype(np.uint8)

        png_path = img_dir / (fits_path.stem + ".png")
        cv2.imwrite(str(png_path), img_8bit)

        h, w = img_8bit.shape

        # Add image entry
        coco_all["images"].append({
            "id": coco_id,
            "file_name": str(png_path),
            "height": h,
            "width": w
        })

        valid_ids.append(coco_id)

        # Add annotations
        if sources is not None and len(sources) > 0:
            for row in sources:

                x = float(row["x_peak"])
                y = float(row["y_peak"])

                x1 = max(0, x - BOX_HALF)
                y1 = max(0, y - BOX_HALF)

                bw = min(BOX_HALF * 2, w - x1)
                bh = min(BOX_HALF * 2, h - y1)

                coco_all["annotations"].append({
                    "id": ann_id,
                    "image_id": coco_id,
                    "category_id": 1,
                    "bbox": [x1, y1, bw, bh],
                    "area": bw * bh,
                    "iscrowd": 0
                })

                ann_id += 1

        coco_id += 1

    except Exception as e:
        print(f"[ERROR] {fits_path.name}: {e}")
        continue

# Final checks
print("\nDataset built:")
print("Images:", len(coco_all["images"]))
print("Annotations:", len(coco_all["annotations"]))

if len(coco_all["images"]) == 0:
    raise RuntimeError("No images were created - check FITS files")

# Split
random.shuffle(valid_ids)

n_val = max(1, int(len(valid_ids) * VAL_SPLIT))
val_ids = set(valid_ids[:n_val])
train_ids = set(valid_ids[n_val:])

def split_coco(coco_full, keep_ids):
    imgs = [im for im in coco_full["images"] if im["id"] in keep_ids]
    anns = [an for an in coco_full["annotations"] if an["image_id"] in keep_ids]

    return {
        "images": imgs,
        "annotations": anns,
        "categories": coco_full["categories"]
    }

coco_train = split_coco(coco_all, train_ids)
coco_val   = split_coco(coco_all, val_ids)

# Saves JSON
train_json = train_dir / "annotations.json"
val_json   = val_dir / "annotations.json"

with open(train_json, "w") as f:
    json.dump(coco_train, f)

with open(val_json, "w") as f:
    json.dump(coco_val, f)

print("\nTrain images:", len(coco_train["images"]))
print("Val images:", len(coco_val["images"]))

c:\Users\dapur\miniconda3\envs\deepdisc\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Looking in: C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\raw\v854_cen_dasch\cutouts
Exists: True
Num FITS files: 4540



Dataset built:
Images: 4540
Annotations: 2199636

Train images: 3859
Val images: 681


In [ ]:
# Pytorch trial run Cell 2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import json
import numpy as np
from PIL import Image
from tqdm import tqdm

# Oaths
BASE_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN")
DATA_DIR = BASE_DIR / "data" / "deepdisc"

TRAIN_JSON = DATA_DIR / "train" / "annotations.json"
VAL_JSON   = DATA_DIR / "val" / "annotations.json"
IMG_DIR    = DATA_DIR / "images"

# Helpers
def crop_to_match(x, ref):
    _, _, h, w = ref.shape
    return x[:, :, :h, :w]

def pad_to_multiple(img, multiple=16):
    _, h, w = img.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple
    return F.pad(img, (0, pad_w, 0, pad_h))

# Dataset
class StarSegDataset(Dataset):
    def __init__(self, json_file):
        with open(json_file, "r") as f:
            self.data = json.load(f)

        self.images = self.data["images"]
        self.anns = {}

        for ann in self.data["annotations"]:
            self.anns.setdefault(ann["image_id"], []).append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info["id"]

        img = Image.open(img_info["file_name"]).convert("L")
        img = np.array(img, dtype=np.float32) / 255.0

        h, w = img.shape
        mask = np.zeros((h, w), dtype=np.float32)

        for ann in self.anns.get(img_id, []):
            x, y, bw, bh = ann["bbox"]
            x, y, bw, bh = map(int, [x, y, bw, bh])
            mask[y:y+bh, x:x+bw] = 1.0

        img = torch.tensor(img).unsqueeze(0)
        mask = torch.tensor(mask).unsqueeze(0)

        img = pad_to_multiple(img)
        mask = pad_to_multiple(mask)

        return img, mask

# Model
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU()
            )

        self.enc1 = block(1, 32)
        self.enc2 = block(32, 64)
        self.enc3 = block(64, 128)

        self.pool = nn.MaxPool2d(2)

        self.mid = block(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = block(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = block(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = block(64, 32)

        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        m = self.mid(self.pool(e3))

        d3 = self.up3(m)
        d3 = self.dec3(torch.cat([d3, crop_to_match(e3, d3)], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, crop_to_match(e2, d2)], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, crop_to_match(e1, d1)], dim=1))

        return torch.sigmoid(self.out(d1))

# Loads data
train_ds = StarSegDataset(TRAIN_JSON)
val_ds   = StarSegDataset(VAL_JSON)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.BCELoss()

# Training loop
EPOCHS = 10

for epoch in range(EPOCHS):

    # Trains model
    model.train()
    train_loss = 0

    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for imgs, masks in train_loop:
        imgs = imgs.to(device)
        masks = masks.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    # Validation tag
    model.eval()
    val_loss = 0

    with torch.no_grad():
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")

        for imgs, masks in val_loop:
            imgs = imgs.to(device)
            masks = masks.to(device)

            preds = model(imgs)
            loss = loss_fn(preds, masks)

            val_loss += loss.item()
            val_loop.set_postfix(val_loss=loss.item())

    # Summary statement
    print(f"\nEpoch {epoch+1}: Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f}\n")

# Saves model
torch.save(
    model.state_dict(),
    BASE_DIR / "data" / "deepdisc" / "train" / "pytorch_trial_1.pth"
)
print("Training complete.")

Epoch 1/10 [Val]: 100%|██████████| 341/341 [00:47<00:00,  7.24it/s, val_loss=0.216] 



Epoch 1: Train Loss=666.5174 | Val Loss=105.6255



Epoch 2/10 [Val]: 100%|██████████| 341/341 [00:42<00:00,  7.96it/s, val_loss=0.19]  



Epoch 2: Train Loss=563.7420 | Val Loss=97.3080



Epoch 3/10 [Val]: 100%|██████████| 341/341 [00:44<00:00,  7.58it/s, val_loss=0.142] 


...


Epoch 8: Train Loss=307.7625 | Val Loss=59.4815



Epoch 9/10 [Val]: 100%|██████████| 341/341 [00:41<00:00,  8.26it/s, val_loss=0.094] 



Epoch 9: Train Loss=296.2151 | Val Loss=55.3974



Epoch 10/10 [Val]: 100%|██████████| 341/341 [00:27<00:00, 12.37it/s, val_loss=0.0937]


Epoch 10: Train Loss=287.9265 | Val Loss=53.5449

Training complete.


In [4]:
# Deepdisc trial run Cell 1